In [ ]:
# MDX Syntax Scan for ESP3D 3.X Docs

This notebook scans `src/content/docs/ESP3D/version-3x` for MDX syntax patterns that may break the Astro/Starlight build.
</VSCode.Cell>
<VSCode.Cell language="python">
from pathlib import Path
import re

root = Path('src/content/docs/ESP3D/version-3x')
mdx_files = sorted(root.rglob('*.mdx'))

print(f'Found {len(mdx_files)} MDX files under {root}\n')
</VSCode.Cell>
<VSCode.Cell language="python">
def suspicious_lines(text):
    lines = text.splitlines()
    results = []
    for i, line in enumerate(lines, start=1):
        if re.search(r'\bESP[0-9]{3}\]', line) and not re.search(r'`\[?ESP[0-9]{3}\]', line):
            results.append((i, line))
        if re.search(r'<[^>]+>', line) and '`' not in line and not line.strip().startswith('```'):
            results.append((i, line))
    return results

matches = {}
for path in mdx_files:
    content = path.read_text(encoding='utf-8')
    found = suspicious_lines(content)
    if found:
        matches[path] = found

print(f'Found {len(matches)} files with suspicious lines')
for path, items in list(matches.items())[:20]:
    print(path)
    for i, line in items[:10]:
        print(f'  {i}: {line}')
    if len(items) > 10:
        print(f'  ... {len(items)-10} more lines')
print()  
</VSCode.Cell>
<VSCode.Cell language="python">
# Show details for the ESP3D commands folder only
for path, items in matches.items():
    if 'documentation\\commands' in str(path):
        print(path)
        for i, line in items:
            print(f'  {i}: {line}')
        print()
</VSCode.Cell>
<VSCode.Cell language="python">
# Collect lines containing raw <...> placeholders outside code spans in the 3.X docs
pattern = re.compile(r'<[^>]+>')
unsafes = []
for path in mdx_files:
    text = path.read_text(encoding='utf-8')
    for i, line in enumerate(text.splitlines(), start=1):
        if pattern.search(line) and '`' not in line and not line.strip().startswith('```'):
            unsafes.append((path, i, line.strip()))

print(f'Raw angle-bracket lines without backticks: {len(unsafes)}')
for path, i, line in unsafes[:50]:
    print(f'{path}:{i}: {line}')
